# Kiểm chứng 4 cấu hình trên Colab

Chọn GPU rồi bấm **Run all**. Notebook tải ảnh và trọng số, tính ba bảng xác minh SKSF-A, xác minh LFW và tìm kiếm SKSF-A từ ảnh gốc.

- Các dòng chính gồm StyleID, ID + Style không trực giao, Disentangle ID512 và Dual-Branch 1280D. Trên LFW, Dual-Branch 1280D áp dụng adapter Hard-Positive cho nhánh ID512.
- Các cặp đánh giá theo giao thức đã khóa của nhóm. SKSF-A có 1.876 cặp, LFW có 6.000 cặp và 10 fold.
- Runtime cần truy cập được các file Drive, Hugging Face và Figshare. Dữ liệu tải về nằm tại `/content/styleid_4models`.

Hai bảng xác minh báo TPR tại FPR tối đa 1%, accuracy tại ba ngưỡng cosine 0.3, 0.4, 0.5 và AUROC. Giá trị cao nhất ở mỗi cột được in đậm.


In [ ]:
# BƯỚC 1: CÀI ĐẶT THƯ VIỆN VÀ KIỂM TRA MÔI TRƯỜNG GPU
import gc
import subprocess
import sys
import torch
print(f"Phiên bản PyTorch: {torch.__version__}")
if not torch.cuda.is_available():
    raise RuntimeError("BẠN CẦN CHỌN GPU: Vào menu Runtime -> Change runtime type -> Chọn T4 GPU rồi chạy lại!")
print(f"--> Đang sử dụng GPU: {torch.cuda.get_device_name(0)}")

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers==4.52.0', 'gdown==6.4.2', 'scikit-learn',
    'pandas', 'tabulate', 'Pillow', 'tqdm',
], check=True)
import numpy as np

torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
print("--> Cài đặt thư viện thành công!")

In [ ]:
from pathlib import Path
import base64
import hashlib
import json
import tarfile
import urllib.request
import zipfile
import zlib

import gdown
import pandas as pd

work_dir = Path('/content/styleid_4models')
work_dir.mkdir(parents=True, exist_ok=True)


def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


def get_gdrive(file_id, output_path, expected_sha256=None):
    def valid(path):
        if not path.is_file():
            return False
        if expected_sha256 is not None:
            return sha256_file(path) == expected_sha256
        return zipfile.is_zipfile(path)

    if valid(output_path):
        return output_path
    partial = output_path.with_suffix(output_path.suffix + '.partial')
    url = f'https://drive.google.com/uc?id={file_id}'
    try:
        gdown.download(url, output=str(partial), quiet=False)
    except Exception as exc:
        raise RuntimeError(f'Không tải được {output_path.name}. Kiểm tra quyền tải và hạn mức tại {url}') from exc
    if not valid(partial):
        raise ValueError(f'File tải về sai định dạng hoặc SHA256: {output_path.name}')
    partial.replace(output_path)
    return output_path


sksf_zip = get_gdrive('1E0QBScC_0cXoMNn7JE1P4f6I9qMFagvz', work_dir / 'SKSF-A.zip')
sksf_dir = work_dir / 'sksf_images'
sksf_dir.mkdir(exist_ok=True)
sksf_hashes = json.loads(zlib.decompress(base64.b64decode('...')))
with zipfile.ZipFile(sksf_zip) as archive:
    members = {}
    for member in archive.infolist():
        if not member.is_dir():
            key = '/'.join(member.filename.replace('\\', '/').split('/')[-2:]).casefold()
            members.setdefault(key, []).append(member)
    for key, expected in sksf_hashes.items():
        matches = members.get(key.casefold(), [])
        if len(matches) != 1:
            raise ValueError(f'Ảnh SKSF-A bị thiếu hoặc trùng: {key}')
        data = archive.read(matches[0])
        if hashlib.sha256(data).hexdigest() != expected:
            raise ValueError(f'SHA256 ảnh SKSF-A không khớp: {key}')
        target = sksf_dir / key
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(data)
print('Đã xác minh đủ 1.072 ảnh SKSF-A.')

print("\n=== [2/4] TẢI MÔ HÌNH KHÔNG TRỰC GIAO (NO-ORTHO - DÒNG 2) ===")
# Drive folder: https://drive.google.com/drive/folders/1W-mlpZHZIANjtu5bPiDTZsHCk9VmRKnE
head_no_ortho = get_gdrive('11nS1fIabk7mLOChF9UW7AU3O3JA0HXoY', work_dir/'disentangled_head_no_ortho.pt', 'def193c21f9b1c072ddf0c7ebb6137b9f45f01d66e1dd20e24feef90ed76579f')
lora_no_ortho = get_gdrive('1TT25NtHy_r_siZ8l-6WJXuqzoC5__eXd', work_dir/'disentangled_lora_clip_no_ortho.pt', '21bcbdcf077071638ce5233a879c3d033c246275a1625af5b791f2cb26a91c2e')

print("\n=== [3/4] TẢI MÔ HÌNH DISENTANGLED CÓ TRỰC GIAO (DÒNG 3) ===")
# Drive folder: https://drive.google.com/drive/folders/11DPH1P4QDKjiZwaAzJsdigK2AAPQezD1
head_ortho = get_gdrive('1ifMm_3hwbYYb6Evr76T8FNtBjnQ69iK6', work_dir/'disentangled_head_SOUP_S6_A100.pt', '5e9dddb7bdf0f45012abd127b83a31b77297b2f5c245cea52b878cfa71ac9968')
lora_ortho = get_gdrive('11RE2F-5JWI3jQymbJ5Z_QTpKSGKpwB6n', work_dir/'disentangled_lora_clip_SOUP_S6_A100.pt', '45fbc2b7e09483c28da2d91d52dbca0114260cfb14e9277c6790b6d93819b24f')

print("\n=== [4/4] TẠO DANH SÁCH CẶP ĐỐI SÁNH VÀ ADAPTER TIÊU CHUẨN ===")
# Nhúng trực tiếp 1.876 cặp SKSF-A đã khóa trong dự án
b64_pairs = 'eJx9ncsOLElSRPd8S4tbGeGPqi9gh+YXQBpmkFqAoDf8PY10I+qaubstQ0dWXo+sjMh4mP3+13/747f//ve//f2P337/l3/96+//8Je//+cf//nj+cf/+o+//fY/f/zv7399/mz85Z//6benovWz0aA9q2xGPqOYayWp1q9vfk1o/Ww0aM8qm5HPKOZaSar965vfE1o/Gw3as8pm5DOKuVaSyn598zah9bPRoD2rbEY+o5hrJan81zfvE1o/Gw3as8pm5DOKuVaSKn598zGh9bPRoD2rbEY+o5hrJany1zefE1o/Gw3as8pm5DOKuVaS6v3rm39PaP1sNGjPKpuRzyjmWkmqz69v/jOh9bPRoD2rbEY+o5hrJameF/RSrxGu0+rgVkpT0BUMVTOLEjvcZ4TrtDq4ldIUdAVD1cyihO73WSNcp9XBrZSmoCsYqmYWJXTGzx7hOq0ObqU0BV3BUDWzKKFrfmyE67Q6uJXSFHQFQ9XMooSO+vERrtPq4FZKU9AVDFUzixK67SdGuE6rg1spTUFXMFTNLEroxJ8c4TqtDm6lNAVdwVA1syihS3/eI1yn1cGtlKagKxiqZhYldPDPZ4TrtDq4ldIUdAVD1UxWLujt12uE67Q6uJXSFHQFQ9XMooTefj0jXKfVwa2UpqArGKpmFiU+bK8RrtPq4FZKU9AVDFUzixJ6+7VHuE6rg1spTUFXMFTNLEro7ZeNcJ1WB7dSmoKuYKiaWZTQ2y8f4TqtDm6lNAVdwVA1syiht18xwnVaHdxKaQq6gqFqZlFCb79yhOu0OriV0hR0BUPVzKKE3n69R7hOq4NbKU1BVzBUzSxK6O3XZ4TrtDq4ldIUdAVD1UxWbujt92uE67Q6uJXSFHQFQ9XMooTefj8jXKfVwa2UpqArGKpmFiX09nuNcJ1WB7dSmoKuYKiaWZQ40b5HuE6rg1spTUFXMFTNLEro7beNcJ1WB7dSmoKuYKiaWZTQ228f4TqtDm6lNAVdwVA1syiht98xwnVaHdxKaQq6gqFqZlFCb79zhOu0OriV0hR0BUPVzKKE3n6/R7hOq4NbKU1BVzBUzSxK6O33Z4TrtDq4ldIUdAVD1UxWGvT29hrhOq0ObqU0BV3BUDWzKKG3t2eE67Q6uJXSFHQFQ9XMooTe3tYI12l1cCulKegKhqqZRQm9ve0RrtPq4FZKU9AVDFUzixIX2W2E67Q6uJXSFHQFQ9XMooTe3nyE67Q6uJXSFHQFQ9XMooTe3mKE67Q6uJXSFHQFQ9XMooTe3nKE67Q6uJXSFHQFQ9XMooTe3t4jXKfVwa2UpqArGKpmFiX09vYZ4TqtDm6lNAVdwVA1k5UOvb2/RrhOq4NbKU1BVzBUzSxK6O39GeE6rQ5upTQFXcFQNbMoobf3NcJ1Wh3cSmkKuoKhamZRQm/ve4TrtDq4ldIUdAVD1cyihN7ebYTrtDq4ldIUdAVD1cyixA12PsJ1Wh3cSmkKuoKhamZRQm/vMcJ1Wh3cSmkKuoKhamZRQm/vOcJ1Wh3cSmkKuoKhamZRQm/v7xGu0+rgVkpT0BUMVTOLEnp7/4xwnVYHt1Kagq5gqJrJyoDePl4jXKfVwa2UpqArGKpmFiX09vGMcJ1WB7dSmoKuYKiaWZTQ28ca4TqtDm6lNAVdwVA1syiht489wnVaHdxKaQq6gqFqZlFCbx82wnVaHdxKaQq6gqFqZlFCbx8+wnVaHdxKaQq6gqFqZlHi5voY4TqtDm6lNAVdwVA1syiht48c4TqtDm6lNAVdwVA1syiht4/3CNdpdXArpSnoCoaqmUUJvX18RrhOq4NbKU1BVzBUzWRlQm+frxGu0+rgVkpT0BUMVTOLEnr7fEa4TquDWylNQVcwVM0sSujtc41wnVYHt1Kagq5gqJpZlNDb5x7hOq0ObqU0BV3BUDWzKKG3TxvhOq0ObqU0BV3BUDWzKKG3Tx/hOq0ObqU0BV3BUDWzKKG3zxjhOq0ObqU0BV3BUDWzKPFgXY5wnVYHt1Kagq5gqJpZlNDb53uE67Q6uJXSFHQFQ9XMooTePj8jXKfVwa2UpqArGKpmsvINvf37NcJ1Wh3cSmkKuoKhamZRQm//fka4TquDWylNQVcwVM0sSujt32uE67Q6uJXSFHQFQ9XMooTe/r1HuE6rg1spTUFXMFTNLEro7d82wnVaHdxKaQq6gqFqZlFCb//2Ea7T6uBWSlPQFQxVM4sSevt3jHCdVge3UpqCrmComlmU0Nu/c4TrtDq4ldIUdAVD1cyixEP17xGu0+rgVkpT0BUMVTOLEnr792eE67Q6uJXSFHQFQ9VMVn6gt/+8RrhOq4NbKU1BVzBUzSxK6O0/zwjXaXVwK6Up6AqGqplFCb39Z41wnVYHt1Kagq5gqJpZlNDbf/YI12l1cCulKegKhqqZRQm9/cdGuE6rg1spTUFXMFTNLEro7T8+wnVaHdxKaQq6gqFqZlFCb/+JEa7T6uBWSlPQFQxVM4sSevtPjnCdVge3UpqCrmComlmU0Nt/3iNcp9XBrZSmoCsYqmYWJRrqfEa4TquDWylNQVcwVM1k5fMiex32uiGDHXa7IYsdoTaNXePQtbNRo9nO65nxus0Wb602jV3j0LWzUaP1zmvNeN1mi7dWm8auceja2ajRiOe1Z7xus8Vbq01j1zh07WzUaMvzshmv22zx1mrT2DUOXTsbNZr0vHzG6zZbvLXaNHaNQ9fORo2WPa+Y8brNFm+tNo1d49C1s1Gjgc8rZ7xus8Vbq01j1zh07WzUaOfzes943WaLt1abxq5x6NrZqNHc58X9N9r7vMhr5/XBDybUprFrHLp2VjUZ+xWvPBx5sM8emfsptWnsGoeunY2abP7Yc4+M/th1j6z+hNo0do1D185GjSMP9tEj2z/24CPjP6U2jV3j0LWzUePIg131yASQHfnIBlCpTWPXOHTtbNQ48mCPPbIEZH8+MgVUatPYNQ5dOxs1jjzYcY8MAtmtjywCldo0do1D185GjSMP9t8ju0D27iPDQKU2jV3j0LWzUePIg934yDyQnfzIPlCpTWPXOHTtbNQ48mBvPrISZF8/MhNUatPYNQ5dOxs1jjzY44+MBdnlj6wFldo0do1D186qRpPBh3z7HrQZfMjz70GjQak2jV3j0LWzUePIg1z8HjQdfMgB8EHbQak2jV3j0LWzUZPdMHv/kuEwu/+S5bBQm8auceja2ahx5EEOfw8aEj7kDvigJaFUm8auceja2ahx5EF+fw/aEz7kFfigQaFUm8auceja2ahx5EHufw+aFT7kHPigXaFUm8auceja2ahx5EFegA9aFz7kI/igeaFUm8auceja2ahx5EHOgA8aGT7kKviglaFUm8auceja2ahx5EE+gQ/aGj7kMfigsaFUm8auceja2ahx5LG4/8aRBzkOPmhzKNWmsWscunZWNRoePuQh+KDl4UP+gw+aHkq1aewah66djRpHHuQo+KAB4kNuhA9aIEq1aewah66djRpHHuQv+KAd4kPehA8aIkq1aewah66djZpiDziDgIIPOIWAog+E2jR2jUPXzkaNIw/yHnzQKvEh38IHzRKl2jR2jUPXzqr+9WP9/P+9KjsZNA26R3sadk+MNuyeE2/YNeZr2LWuPQxtxG1i3+neBt4kl4adMJoGXYfnhl0DgobdM9WH0d9mhPw7oOfImmTfWfgGXu+Bhn1nZBp4TyUdhtbhE7qWbw37Tro38G5WbJiL93Kt6xp2nSAOw/3J74ldR+eGbb4qcLvSZ2LXZLthdwtZw+4WxcPw0PFnYjd6pmHfyZUGXnfaht0dMQ27pvoNu8d2DsNNSjax647UsBs+0bCbntOw6/vQsHuMqmE3F+AwPGf8TOw6nTds8wWDx472xL4Tdw28W98adl0DD8N9STGxu0e0Ydd5p2H3TG7DbnBGw67JXsOy9DvoErpH+J2a62j5MXBFMJ8Rlpssrga+fax543IuRCuR1whP9lrH7hGdDn53fHTU+Qej9b/3WPQ6CF3YHTfq4DVD6uD3cayj3305HfVyn8NFv/0eyybfWnHBb+cIr29FB7d62e/ibEe9/m64C9nGqlmuT1Mfhoa7+MenFb41v7BxT0nLe3uE10e/g9eO9kJ0C10jrINEWtV7j9LvVsSOerlz0LTaM1a9ThwXoj34yG4oQAd3+RfjuaPPCL08J+Ap4xxrfteSL031w9DqHf0VsaMflVb+Tzh7xh+mcwztYJZxOq7Z8S2pO2jcwS1f13iUj4t1fN1jn59j0dLp0zLdyO6Jww5+ZzY7eoJtO1avMdyMPNe8p3PvYxeaiO0R3sy/Dm6lNB73UfjXrPzOO3c0+X6DS3HbRnjjGjr4XdTp6HVw7qDzn4lW4PgJW3X+uPzme4T32HQHd+nzcOXtkyP87h/p6HcqpaPfJftLsfdfIyy9Py640awILbfNNZ2fVnGpLcaSyWNIXGWjcQFFgOGrYsf/GYXGY1pcXBPvp4wvcWGNSuLDfIzwRhh28DsT2dFrVdfBMivBi2kxVq1/F+z1fYTfJd2Obn404kW0PdLvvuGOXvfjDmaZKMMUsBl+9/x19J497KCVySRaOHvlSOuQjJbN6EvErj9GeNNYO/jdr95RK39wjAL7jDDKHQcNRLF7x3Uyf41wlclBXCSjhwZaIuOfBg3D3yO8Rkkd/G5fORQXx8JHeLM/O/jdz9dR41+cV8VGGD9srJk8mY3LYXTbwcUwo5elg0cxSr/L6B39bg/oaPBDOq2CUT9Ni2Axwvpx8PTxZ1QajVxp7es102td18Esc2u47pU+wvqASYte/O1jTMgzQud/Kq530VMZrnbRFDitdfkIy80M17loPodWueaXrf+Ydm9NR7M8z2316I+xYDT83zQGmKU3MbeDN/2gg2U4g7lgfJXhxP4a4fdcVEdvHmAHrdwjaf/uCG/6bwezrkCh58ga4TXh7OC1p++gqZd1fqLDZDD++9NBIZJiNJiNsDz5YzQYz0ViNhh/GJztn5XxY481s9bEJNAYYRk0YzaYz0qrS5ydpWgHgx9sMRzsgy+L4WD0b8JwsMCXxXAwWvXAcLC0ETrfcDAc7JxH6Og1TLsQn/1HVi4xDAejCRkMB6OnClNP/hgORrOxGA5GM8AUDvYa4XVH7eAuXzzGhfgIvawj47K9jTXLlD+Gg/FVjelgDy3yYTwYzfFiPBg9bWA82JphnZHBfDDauYH5YM8M1w/6XbDfz1FofHPEeDD+dnERf401swzPMR+MF00wIAzvjpgPRk9VmA9GjxqYD7ZnZfzgXwVX8+lSwAf/9wjrozIGhNEsBgaE8TeERmOvEZalZKMJf1Tiqv4e4Q1y6eDmMQoGhPEtDq1H1gi/B587muVOhav7a4R1wwsmhMWotDJHgRFhM7u5bR3M8hel48AzLZOXGBFGa0oUEfaM0Esf0ZmPdDB55xZGhD00M08ZYbSHBzPC9qi0MmrCkDAahmBI2Ps1Fi1TsRQS9hnh+oGdC2WEza9a1/oxJIxmTTEkjGZiMCSMhlsYEkbzohgS9sHeBUPC+EEMU8LKh1nqw6C56Fg0eTUKU8LiPcLFf0NMCaP+DlPC+LrGTn+u+T3Q29GyWQxjwugZA2PCyuua/GGw35+relk5x6Qw+4xVb7TDhXSQZoTNp8Hn/VFpPGGKSWEfH2GUDtrV4z5GhfG1iwaj9LLY769ReY2rO+hlQItZYbQq5/S4T+8XH/d9hHXuAtPCaJsgpoXREIfSwl4j/BpNdDTLpk3o+s1HeH24O1in+zEv7BmZ88MhxoWljzWT50swLuzhf4xa6se8sIfmwJym+32kLmnwSBAjw/iuE9T975GWaWUMDVvz65Zn/qD5/rnoNbPtYJnxDzXjj6lh9GfE1DBaw8XUMLqbYWrYnpVBQw4MDaMLiULDbIR19RJTw3hXMhl8vEf69dDpaFm9xNww7qAwOIy3uGFy2PG76mhZjcXoMP72cQgwsuDN6ZgcxktvGB1GiwEYHUajmaABgI1SKwtzoWb8KTzMx6rJvVvQg/97pOtMtndw1/eLFuN7hPV2hvlhD+3KwwAxnqrABLGHlskwQoyOOWCEWLkV0uN/jtR5hEshYjZWLTNMGCLGi12UIkZnBXAUML/u1zWmo2VLOcaIUddJMWL002CwyAxXuemj1fgelWVlCWPEdowwzsx4B8tzZtAEwEzXmaXuYNnnjzlifI2REdh7pHVmBpPEaGycNAZ4Rvp1bevo5lVpzBKjCQLMEuPlLgwTO3acHf3zx0ElGo5/RrjK0iimia1RaTwtRmFi7xE2H4UW/en94qa/HOEqu6MwT0xIy4kSzBPjPcgYKKaq/nmhoZK8RZ+R3rCBDtbnAIwUo625GClWrm4YB+w9Vk2aPcRIseOj2lGea8JIsTUL6+kFzBSj5SfMFONzORgqZvRhcBSwR3jDKjtY9pZgqBg9xGKomM/wa5rU0Syvi9MA7xEunielVLE1Kuv5UIwVi/kdlVV/jBXj+VfMFfORrR+0NoqxYjTBirFi9LSDsWLPDOMcVupg+e/j7L+NcJ2pxA5uvtdD308LDhgqRk+DGCrG/3tyxqD/A3T+NFGHqWK0wSvlDADGitECJ8aKfWYYZ82yg2VmFmPFaPWdYsXwx8ZYMXqiw1ixHNmN4urg9YPvYN1UgrFi9JSOsWJ0a8RYsZhftqyVvVW/j7Fi9BSDsWJ0m8dYMZqAxlixh/6DmCtG1wnmivF5VzK6mN/SzbLp4J99Cwqp0/+MtC75Y7IYX2N0zm+N1Pk89Jum/z9j1bKKidliPDZ6q6d/DBfL+XWtbBvFdDG+cWC8GM/TY74YLaFQvtge4U107+DmMzuYL0Y96Zse/nOkoYrWvcuYMMbnwHEBgO4smDA2Cq3MXVHCmI2wHI57U8dPf1Xs+H2E5Rw5RozReg9FjH1G6OVXw+f+Z6yZtNkKE8Z40gsjxmjpCiPG2CwA+v6YYT2HgRlj5a6DT/70wuSKtUfa3M7IjTNGLZ/1w5gxmvXGmDGyW8CYMXoKxJgxfqrCnDH6J2LOGC2vUM5YjNDLN4hBY9TPYNAYz8ti0hg9FVDSGFkvkN33e5RW9wUcAXxGWKZlMWqMJpgwaoz6Yowao4l4jBp7z0pTL1uOlHzU5D9GjdFO1g/N/ftIy/5FzBqjPRCYNbbmomUlE7PG6A+DWWO0hYSyxvYIS+ePWWM0UsSsMRuZ84ItRo3xt0cemqjEJ34b4eI5Xowa4zdE3pk5UuexNGaN2Viz2Chh1Bg/QWPWGM/4YdgYT4Rj2hhNJWDaGP9d0CxzLpo8WfChKX8b6eJdxBg3RuMqjBvjfT+YN1Z+N3zq/4xVy0E/DByjPZ4YOEbz7xg4xn8YnPNfI3SeasDAMXoow8AxnmHHxDGf4eIH1w91/DlK68o/Zo7lLI3yLIihY9Q3cehYzHT9oEleihyjLRKUOEZjVwoco6Uyyhtjmw+MG2P3Bkwbo4EbhY2RGRJljdGDBUWNqbrO5g+cM8bfJM0DkP8D5Ywt/ky4+58/E84FzFIrtkiYMZb8A8NggHY1csIYe8BQwhi/LTz5z95SuBdAvHLZp8HpYg9/YlwIEIWr9w9mi23+SDAuoINSlCzGxkEULCZe2etfGHcF8q8Eo4Nyc4DhQfn9aWMg2Ytgptjma5aGCPwzoR8Af9N0LEBUzuLYQ4FibJlCgWK0J4QDxR7+TmCsUD6VOhXIcWK06ZHixGiShtLEgu+2MGAgRxuMElv8iXCVgK8QPBs4w3I2kFPE+MtQmwU5Q2zxV0XOQPyRKMlDvHiZAKEIMbY6ogQxNm6iBDH2+cEEscWXNa0b8BvDzFK+rGnbAH9mGEK8uXv6qEsPbQL5M6FPIJ3poOwwelrn6DCav+HoMLZMQ7PAtLlyXUbg5DD27sMNBMW9D4cR4qWNn80pNoz/yOga+N5z3bLIw5lhxayQ/LP5tXFD4aytm9Y4Maz8jriXgH8oXFagGxDlhfFfHQ0EP/yZyEVAvHZznyATQVE5zmGCllZLRLIR5FfGXQX8dZExMF98tMLAPwXOM7DbJYwl+KLHFQaWoqMAXz8wlKCzPpQRxt9Gm03a4jJnSgFhzp8IxhHl70QTDgJXT0E0FTShtea6hGFE8ifGYQTbTKK1YDElbUNJW/yNVGlx00ORwSBfmZQMxndzGEywwyh6DLJrJHoMllsIDibKFwaDic3fNo4l+E3jUIKvelyAeOVcOHneiSPB2P0Q7QZ57EV+g7O0mCVTHFjyBUKZHPyRcBcCe8hSDKnPuLn40HiQrTjReZC2PlIUGLnfcRLYk3PhOjpC90G+9tB+kDbvUgwYX9XoP8jP7hQCxn0uh4DNhbNaDNN0BH9iPJbIXsDKhogCwMoFQvlf7xmX0+IU/7X5TdM4gr9N3JbIlx7uUOCvmoYRonIxs+for3Lt4S4F0ko3Ysr9evFngnEE91DoS8h9H6d+PTMO9qmh0K83v2tcsXjxpYlpA9TrokEhP22gQeGHLy4YSJS/hLQo5Lwv/rpwJMGFYSAR/JFwtyJfATCQ4Id3tCnksdHS0xFkVMiXLToVCtrMHKFXIXvsolkhd0BoVshfJXkVzmWTN3FwwhdfO3h8kX9+jCbgaweGEB/+riiegP9LuJLB/yXcxMDfJJ1f4Mp4gIHvPDCEMP620MOAvy3cyMAfCV0M+CNhYgF9l5TqxT0EWhdyXA56F3KHSplePAuL7oVsPk6RXjxhgA6GRSxMjDjPi3MY0MSQ7w9brmiQiSF/YDIxZpt8HEKwGH0N2PUfpyLokDFFefEUCVkZ8loKuhnyzDK6GVJICuV48diFYrz4foqGhjyvjIaGwb8EDCG4a0JHQ769kKVhzHXz/ylqYQQh4LoLPi3e5dkHfQ35mpaBBmxsyN8kLmn8/BX+D5TFNb8='
pairs_csv_text = zlib.decompress(base64.b64decode(b64_pairs)).decode('utf-8')
(work_dir/'sksf_pairs.csv').write_text(pairs_csv_text, encoding='utf-8')
print(f"  + Đã nạp danh sách 1.876 cặp SKSF-A đã khóa.")

# Nhúng trực tiếp trọng số Adapter Hard-Positive (32.768 params)
b64_down_data = '...'
b64_up_data = '...'
w_down = np.frombuffer(zlib.decompress(base64.b64decode(b64_down_data)), dtype=np.float16).astype(np.float32).reshape(32, 512)
w_up = np.frombuffer(zlib.decompress(base64.b64decode(b64_up_data)), dtype=np.float16).astype(np.float32).reshape(512, 32)
np.savez(work_dir/'hp_adapter_weights.npz', w_down=w_down, w_up=w_up)
print(f"  + Đã sẵn sàng trọng số Hard-Positive Adapter!")

In [ ]:
# BƯỚC 3: ĐỊNH NGHĨA KIẾN TRÚC MẠNG NƠ-RON (LORA CLIP, IDENTITY HEAD, RESIDUAL ADAPTER)
import torch.nn as nn
import torch.nn.functional as F

CLIP_DIM = 768
IDENTITY_DIM = 512
LORA_RANK = 8
LORA_ALPHA = 1.0

class LoRALinear(nn.Module):
    def __init__(self, linear_layer: nn.Linear, r: int = 8, alpha: float = 1.0):
        super().__init__()
        self.linear = linear_layer
        self.scaling = alpha / r
        self.lora_down = nn.Linear(linear_layer.in_features, r, bias=False)
        self.lora_up = nn.Linear(r, linear_layer.out_features, bias=False)
        nn.init.kaiming_uniform_(self.lora_down.weight, a=5**0.5)
        nn.init.zeros_(self.lora_up.weight)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.linear(x) + self.lora_up(self.lora_down(x)) * self.scaling

def apply_lora_to_clip(model: nn.Module):
    for layer in model.vision_model.encoder.layers:
        for name in ('q_proj', 'k_proj', 'v_proj', 'out_proj'):
            linear = getattr(layer.self_attn, name)
            setattr(layer.self_attn, name, LoRALinear(linear, LORA_RANK, LORA_ALPHA))

class IdentityHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.id_proj = nn.Sequential(
            nn.Linear(CLIP_DIM, IDENTITY_DIM),
            nn.BatchNorm1d(IDENTITY_DIM),
            nn.PReLU(),
        )

    def forward(self, features: torch.Tensor) -> torch.Tensor:
        return self.id_proj(features)

class ResidualMetric(nn.Module):
    """Residual Bottleneck Adapter (32.768 params) cho Hard-Positive Alignment"""
    def __init__(self, dimension=512, rank=32, limit=0.25):
        super().__init__()
        self.limit = limit
        self.down = nn.Linear(dimension, rank, bias=False)
        self.up = nn.Linear(rank, dimension, bias=False)

    def forward(self, x):
        delta = self.up(torch.tanh(self.down(x)))
        norm = torch.clamp(delta.norm(dim=1, keepdim=True), min=1e-12)
        delta = delta * torch.clamp(self.limit / norm, max=1.0)
        return F.normalize(x + delta, dim=1)

# Nạp module Hard-Positive Adapter
hp_adapter = ResidualMetric().to('cuda').eval()
with np.load(work_dir/'hp_adapter_weights.npz') as hp_weights, torch.no_grad():
    hp_adapter.down.weight.copy_(torch.from_numpy(hp_weights['w_down']).to('cuda'))
    hp_adapter.up.weight.copy_(torch.from_numpy(hp_weights['w_up']).to('cuda'))

print("Đã nạp kiến trúc mạng và trọng số Adapter thành công!")

In [ ]:
# BƯỚC 4: HÀM MỞ ẢNH AN TOÀN VÀ TRÍCH XUẤT ĐẶC TRƯNG TRÊN GPU
from PIL import Image
import numpy as np
from tqdm import tqdm

def open_img_safe(base_dir, rel_path):
    with Image.open(base_dir / rel_path) as image:
        return image.convert('RGB')

def extract_features(model, head, processor, img_paths, base_dir, batch_size=32, desc="Inference"):
    model.eval()
    if head is not None: head.eval()
    vecs = []
    for idx in tqdm(range(0, len(img_paths), batch_size), desc=desc):
        batch_keys = img_paths[idx:idx + batch_size]
        batch_imgs = [open_img_safe(base_dir, k) for k in batch_keys]
        try:
            inputs = processor(images=batch_imgs, return_tensors='pt').to('cuda')
        finally:
            for image in batch_imgs:
                image.close()
        with torch.no_grad():
            pooled = model.vision_model(pixel_values=inputs['pixel_values']).pooler_output
            features = model.visual_projection(pooled)
            emb = head(features) if head is not None else features
            emb = emb / emb.norm(dim=-1, keepdim=True)
        vecs.append(emb.cpu().float().numpy())
    vectors = np.concatenate(vecs, axis=0)
    if not np.isfinite(vectors).all() or not np.allclose(np.linalg.norm(vectors, axis=1), 1, atol=1e-5):
        raise ValueError(f'Embedding không hợp lệ: {desc}')
    return vectors.astype(np.float64)

print("Hàm trích xuất đặc trưng đã sẵn sàng!")

In [ ]:
# Chuẩn bị LFW View 2 trước khi nạp model.
LFW_URL = 'https://ndownloader.figshare.com/files/5976015'
LFW_SHA256 = 'b47c8422c8cded889dc5a13418c4bc2abbda121092b3533a83306f90d900100a'
lfw_dir = work_dir / 'lfw_funneled'
lfw_archive = work_dir / 'lfw_funneled.tgz'
(work_dir / 'lfw_pairs.csv').write_bytes(zlib.decompress(base64.b64decode('...')))
df_lfw = pd.read_csv(work_dir / 'lfw_pairs.csv')
unique_lfw_images = sorted(set(df_lfw['left']) | set(df_lfw['right']))
if len(df_lfw) != 6000 or len(unique_lfw_images) != 7701:
    raise ValueError('Danh sách LFW phải có 6.000 cặp và 7.701 ảnh.')
if df_lfw.groupby(['fold', 'label']).size().to_dict() != {
    (fold, label): 300 for fold in range(10) for label in (0, 1)
}:
    raise ValueError('LFW phải có 10 fold, mỗi fold gồm 300 cặp dương và 300 cặp âm.')
if not lfw_archive.is_file() or sha256_file(lfw_archive) != LFW_SHA256:
    partial = lfw_archive.with_suffix('.tgz.partial')
    print('Đang tải archive LFW funneled từ Figshare...', flush=True)
    urllib.request.urlretrieve(LFW_URL, partial)
    if sha256_file(partial) != LFW_SHA256:
        raise ValueError('SHA256 archive LFW không khớp.')
    partial.replace(lfw_archive)
remaining = set(unique_lfw_images)
with tarfile.open(lfw_archive, 'r|gz') as archive:
    for member in archive:
        key = '/'.join(member.name.split('/')[-2:])
        if member.isfile() and key in remaining:
            target = lfw_dir / key
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.extractfile(member) as source:
                target.write_bytes(source.read())
            remaining.remove(key)
if remaining:
    raise FileNotFoundError(f'Archive LFW thiếu {len(remaining)} ảnh, ví dụ {min(remaining)}.')
print('Đã xác minh đủ 6.000 cặp LFW, 10 fold và 7.701 ảnh.')


In [ ]:
from transformers import CLIPModel, CLIPProcessor

STYLEID_REVISION = '1967c354f339a636e5b3e16ecab3d0075aa27ab1'
BASE_REVISION = '32bd64288804d66eefd0ccbe215aa642df71cc41'
all_sksf_keys = sorted(sksf_hashes)
processor = CLIPProcessor.from_pretrained('kwanY/styleid', revision=STYLEID_REVISION)


def encode_both(name, backbone_path=None, head_path=None):
    repository = 'kwanY/styleid' if backbone_path is None else 'openai/clip-vit-large-patch14'
    revision = STYLEID_REVISION if backbone_path is None else BASE_REVISION
    model = CLIPModel.from_pretrained(repository, revision=revision,
                                     torch_dtype=torch.float32, attn_implementation='eager')
    head = None
    if backbone_path is not None:
        apply_lora_to_clip(model)
        model.load_state_dict(torch.load(backbone_path, map_location='cpu', weights_only=True), strict=True)
        head = IdentityHead()
        state = torch.load(head_path, map_location='cpu', weights_only=True)
        head.load_state_dict({key: state[key] for key in head.state_dict()}, strict=True)
        head = head.to('cuda').eval()
    model = model.to('cuda').eval()
    sksf = extract_features(model, head, processor, all_sksf_keys, sksf_dir, desc=f'{name} SKSF-A')
    lfw = extract_features(model, head, processor, unique_lfw_images, lfw_dir, desc=f'{name} LFW')
    del model, head
    gc.collect()
    torch.cuda.empty_cache()
    return sksf, lfw


vecs_styleid_sksf, lfw_st_vecs = encode_both('StyleID')
vecs_no_ortho_sksf, lfw_no_ortho_vecs = encode_both('No-Ortho', lora_no_ortho, head_no_ortho)
vecs_ortho_sksf, lfw_ortho_vecs = encode_both('Disentangle', lora_ortho, head_ortho)
vecs_dual_sksf = np.concatenate([
    np.sqrt(0.5) * vecs_styleid_sksf, np.sqrt(0.5) * vecs_ortho_sksf
], axis=-1)


In [ ]:
# Bảng 1. Xác minh SKSF-A
from IPython.display import Markdown, display
from sklearn.metrics import roc_curve, roc_auc_score
from tabulate import tabulate

ACC_THRESHOLDS = (0.3, 0.4, 0.5)
VERIFICATION_COLUMNS = [
    'TPR@1% FPR ↑',
    *(f'Acc@{threshold:g} ↑' for threshold in ACC_THRESHOLDS),
    'AUROC ↑',
]


def verification_metrics(labels, scores):
    fpr, tpr, _ = roc_curve(labels, scores, drop_intermediate=False)
    tpr_at_1pct_fpr = tpr[fpr <= 0.01].max()
    accuracies = [np.mean((scores >= threshold) == labels) for threshold in ACC_THRESHOLDS]
    return (tpr_at_1pct_fpr, *accuracies, roc_auc_score(labels, scores))


def display_comparison(title, columns, results):
    best_by_column = [max(values) for values in zip(*(metrics for _, metrics in results))]
    rows = []
    for method, metrics in results:
        cells = []
        for value, best in zip(metrics, best_by_column):
            formatted = f'{value:.4f}'
            cells.append(f'**{formatted}**' if value == best else formatted)
        rows.append([method, *cells])

    table = tabulate(rows, headers=['Methods', *columns], tablefmt='github',
                     disable_numparse=True, colalign=('left',) + ('right',) * len(columns))
    display(Markdown(f'### {title}\n\n{table}'))


df_pairs_sksf = pd.read_csv(work_dir / 'sksf_pairs.csv')
key2idx_sksf = {key: i for i, key in enumerate(all_sksf_keys)}

if len(df_pairs_sksf) != 1876 or df_pairs_sksf['label'].value_counts().to_dict() != {0: 938, 1: 938}:
    raise ValueError('SKSF-A phải có đúng 938 cặp dương và 938 cặp âm.')
l_sksf = np.array([key2idx_sksf[key] for key in df_pairs_sksf['left']])
r_sksf = np.array([key2idx_sksf[key] for key in df_pairs_sksf['right']])
y_sksf = df_pairs_sksf['label'].values

models_sksf = [
    ('StyleID', vecs_styleid_sksf),
    ('ID + Style không trực giao', vecs_no_ortho_sksf),
    ('Disentangle ID512', vecs_ortho_sksf),
    ('Dual-Branch 1280D', vecs_dual_sksf),
]
sksf_results = []
for name, vectors in models_sksf:
    scores = (vectors[l_sksf] * vectors[r_sksf]).sum(axis=-1)
    sksf_results.append((name, verification_metrics(y_sksf, scores)))

display_comparison('Bảng 1. Xác minh SKSF-A (1.876 cặp)', VERIFICATION_COLUMNS, sksf_results)


In [ ]:
# Áp dụng adapter Hard-Positive cho nhánh ID512 của Dual-Branch trên LFW.
with torch.inference_mode():
    lfw_hp_vecs = hp_adapter(torch.from_numpy(lfw_ortho_vecs).float().to('cuda')).cpu().numpy().astype(np.float64)


In [ ]:
# Bảng 2. Xác minh LFW
lfw_k2i = {k: i for i, k in enumerate(unique_lfw_images)}
l_lfw = np.array([lfw_k2i[k] for k in df_lfw['left']])
r_lfw = np.array([lfw_k2i[k] for k in df_lfw['right']])
labels_lfw = df_lfw['label'].values
folds_lfw = df_lfw['fold'].values

# Điểm cosine tương đồng từ các vector
scores_st = (lfw_st_vecs[l_lfw] * lfw_st_vecs[r_lfw]).sum(axis=-1)
scores_no_ortho = (lfw_no_ortho_vecs[l_lfw] * lfw_no_ortho_vecs[r_lfw]).sum(axis=-1)
scores_ortho = (lfw_ortho_vecs[l_lfw] * lfw_ortho_vecs[r_lfw]).sum(axis=-1)
scores_hp = (lfw_hp_vecs[l_lfw] * lfw_hp_vecs[r_lfw]).sum(axis=-1)

scores_dual_lfw = 0.5 * scores_st + 0.5 * scores_hp

def lfw_cross_validation(scores):
    grid = np.linspace(-0.2, 0.95, 200)
    fold_accs = []
    for fold in range(10):
        train = folds_lfw != fold
        test = folds_lfw == fold
        best_threshold = max(
            grid, key=lambda threshold: np.mean((scores[train] >= threshold) == labels_lfw[train])
        )
        fold_accs.append(np.mean((scores[test] >= best_threshold) == labels_lfw[test]))
    return np.mean(fold_accs), np.std(fold_accs)


models_lfw = [
    ('StyleID', scores_st),
    ('ID + Style không trực giao', scores_no_ortho),
    ('Disentangle ID512', scores_ortho),
    ('Dual-Branch 1280D', scores_dual_lfw),
]
lfw_results = [(name, verification_metrics(labels_lfw, scores)) for name, scores in models_lfw]
display_comparison('Bảng 2. Xác minh LFW View 2 (6.000 cặp)', VERIFICATION_COLUMNS, lfw_results)

cv_lines = []
for name, scores in models_lfw:
    mean, std = lfw_cross_validation(scores)
    cv_lines.append(f'- {name} — {mean:.2%} ± {std:.2%}')
display(Markdown('**Độ chính xác 10-fold LFW**\n\n' + '\n'.join(cv_lines)))


In [ ]:
# Bảng 3. Truy vấn danh tính SKSF-A
gallery_keys = [k for k in all_sksf_keys if k.startswith('Photo/')]
query_keys = [k for k in all_sksf_keys if not k.startswith('Photo/')]
if len(gallery_keys) != 134 or len(query_keys) != 938:
    raise ValueError('SKSF-A retrieval phải có 134 ảnh gallery và 938 truy vấn.')
gallery_ids = np.array([int(Path(k).stem) for k in gallery_keys])
g_idx = [key2idx_sksf[k] for k in gallery_keys]

retrieval_results = []
for name, vectors in models_sksf:
    ranks = []
    for qk in query_keys:
        target_id = int(Path(qk).stem)
        q_vec = vectors[key2idx_sksf[qk]]
        sims = vectors[g_idx] @ q_vec
        true_pos = np.flatnonzero(gallery_ids == target_id)[0]
        sorted_idx = np.argsort(-sims, kind='stable')
        rank = int(np.flatnonzero(sorted_idx == true_pos)[0]) + 1
        ranks.append(rank)
    ranks = np.array(ranks)
    r1 = np.mean(ranks == 1)
    r5 = np.mean(ranks <= 5)
    mrr = np.mean(1.0 / ranks)
    retrieval_results.append((name, (r1, r5, mrr)))

display_comparison(
    'Bảng 3. Truy vấn danh tính SKSF-A (134 ảnh gallery, 938 truy vấn)',
    ['Rank-1 ↑', 'Rank-5 ↑', 'MRR ↑'],
    retrieval_results,
)
